# Andes Capital Real Estate | Desempeño comercial inmobiliario

El caso parte de una necesidad comercial: convertir los registros de ventas en una visión de conjunto. Analicé el periodo 2023–2024 para entender cómo evolucionaron los ingresos, qué tipos de propiedad, canales y segmentos de clientes aportaron más, y cuándo se registraron nuevas compras después de la primera.

Validé los datos en Power Query y organicé las ventas, los clientes, las propiedades y el calendario en un modelo estrella. Con DAX calculé indicadores de ventas e ingresos, participaciones y crecimiento interanual. Presenté el análisis en un dashboard de Power BI con tres vistas: resumen ejecutivo, análisis comercial y cohortes de clientes.

# 📂 Descripción de los Datasets

🔹 **hecho_ventas_propiedades**  
Tabla de hechos con las transacciones de venta (precio, cliente, propiedad, canal, fecha).

🔹 **dim_clientes**  
Información y segmentación de clientes.

🔹 **dim_propiedades**  
Características de las propiedades (tipo, tamaño, ubicación, etc).

🔹 **dim_fecha**  
Tabla calendario que deberá ser creada como parte del modelado.

# 🎯 Preguntas que el dashboard debe responder

**Desempeño general**

- ¿Cuál es el **ingreso total** generado por la empresa?
- ¿Cuántas **propiedades se han vendido**?
- ¿Cuál es el **precio promedio de venta**?
- ¿Cuál es la **comisión total** generada por la empresa?

**Análisis comercial**

- ¿Qué **tipo de propiedad genera más ingresos**?
- ¿Qué **segmento de clientes compra más propiedades**?
- ¿Qué canal de venta **genera mayor ingresos**?


**Análisis temporal**

- ¿Cómo evolucionan las **ventas en el tiempo**?
- ¿El negocio está **creciendo año contra año**?
- ¿Cómo va el desempeño **acumulado del año actual (YTD)**?

**Cohortes de clientes**

- ¿Los clientes **vuelven a comprar** después de su primera compra?
- ¿Qué **cohortes de clientes generan más compras/ingresos con el tiempo**?

## Paso 1 · Validación de calidad de datos

- **Tipos de datos.** En `hecho_ventas_propiedades`, `fecha_venta` quedó como fecha con configuración regional `es-MX`, `precio_venta` y `monto_comision` como números decimales, y `porcentaje_comision` como porcentaje. Los identificadores se conservaron como texto.
- **Valores vacíos y errores.** El perfil de Power Query mostró 0 % de vacíos y 0 % de errores en las columnas revisadas de `hecho_ventas_propiedades`, `dim_propiedades` y `dim_clientes`.
- **Duplicados en las claves.** En `dim_clientes`, `id_cliente` mostró 3,500 valores distintos y 3,500 únicos. En `dim_propiedades`, `id_propiedad` mostró 8,000 distintos y 8,000 únicos. En ambos casos, cada identificador registrado aparece una sola vez. En la tabla de ventas, `id_venta` también mostró 8,500 valores distintos y 8,500 únicos.

## Paso 2 · Tabla calendario en Power BI

La tabla `dim_fecha` contiene una fila por día, desde la fecha mínima hasta la fecha máxima de `hecho_ventas_propiedades[fecha_venta]`. El rango se calcula a partir de los datos de ventas. `ADDCOLUMNS` agrega el año, el nombre del mes, el número de mes y el periodo `Año-Mes`. En el modelo actual, el calendario contiene 731 filas.

```dax
dim_fecha =
ADDCOLUMNS(
    CALENDAR(
        MIN('hecho_ventas_propiedades'[fecha_venta]),
        MAX('hecho_ventas_propiedades'[fecha_venta])
    ),
    "Año", YEAR([Date]),
    "Mes", FORMAT([Date], "MMMM"),
    "Número de mes", MONTH([Date]),
    "Año-Mes", FORMAT([Date], "yyyy-MM")
)
```

## Paso 3 · Modelo de datos

Construí un esquema estrella con `hecho_ventas_propiedades` como tabla central y tres dimensiones: `dim_clientes`, `dim_propiedades` y `dim_fecha`. Las relaciones activas de uno a muchos permiten analizar las transacciones por comprador, propiedad y periodo.

## Paso 4 · Cálculos DAX del reporte

#### Crecimiento interanual

La medida compara los ingresos del periodo seleccionado con los del mismo periodo del año anterior. La diferencia se divide entre los ingresos del periodo anterior.

```dax
Ventas Año Anterior =
CALCULATE(
    [Ingreso Total],
    DATEADD(
        'dim_fecha'[Date],
        -1,
        YEAR
    )
)

Crecimiento YoY =
DIVIDE(
    [Ingreso Total] - [Ventas Año Anterior],
    [Ventas Año Anterior]
)
```

#### Participación en los ingresos

Cada medida divide el ingreso de la categoría entre el total calculado al retirar el filtro de esa categoría. El filtro de año se conserva en el cálculo.

```dax
Participación Ingresos por Tipo =
DIVIDE(
    [Ingreso Total],
    CALCULATE(
        [Ingreso Total],
        ALL('dim_propiedades'[tipo_propiedad])
    )
)

Participación Ingresos por Canal =
DIVIDE(
    [Ingreso Total],
    CALCULATE(
        [Ingreso Total],
        ALL('hecho_ventas_propiedades'[canal_venta])
    )
)

Participación Ingresos por Segmento =
DIVIDE(
    [Ingreso Total],
    CALCULATE(
        [Ingreso Total],
        ALL('dim_clientes'[segmento_comprador])
    )
)
```
**Validación con el filtro de año.** Para 2024, dividí los ingresos de Casa, Corredor y Primera vez entre el ingreso total del año (3,164,799,714.00). Los resultados coincidieron con los porcentajes del dashboard a dos decimales (36.24 %, 72.89 % y 62.74 %, respectivamente).

#### Columnas calculadas para cohortes

La primera fecha de compra asigna cada cliente a una cohorte. `Mes Cohorte` identifica el mes de esa primera compra y `Mes Venta` identifica el mes de cada transacción.

```dax
Primera compra por cliente =
CALCULATE(
    MIN('hecho_ventas_propiedades'[fecha_venta]),
    ALLEXCEPT(
        'hecho_ventas_propiedades',
        'hecho_ventas_propiedades'[id_cliente]
    )
)

Mes Cohorte =
FORMAT(
    'hecho_ventas_propiedades'[Primera compra por cliente],
    "YYYY-MM"
)

Mes Venta =
FORMAT(
    'hecho_ventas_propiedades'[fecha_venta],
    "YYYY-MM"
)
```

## Paso 5 · Dashboard en Power BI

- **Resumen ejecutivo.** Reúne ingresos, número de ventas, ticket promedio y comisiones. Compara la evolución mensual de los ingresos con el año anterior.
- **Análisis comercial.** Compara los ingresos por tipo de propiedad, canal de venta y segmento de cliente. Los ejes muestran los importes en millones y las etiquetas indican la participación en el ingreso seleccionado.
- **Análisis de cohortes.** Agrupa a los clientes por el mes de su primera compra y muestra cuántas transacciones registró cada cohorte por mes de venta.

## Paso 6 · Resumen ejecutivo

En 2024 se registraron **4,455 ventas**, con ingresos de **3,164,799,714.00**, un ticket promedio de **710,392.75** y comisiones de **105,891,941.00**. Los ingresos crecieron **11.14 %** frente a 2023. Bogotá registró el mayor volumen de ventas, con **2,303 operaciones**.

### Hallazgos y decisiones comerciales

- **Tipo de propiedad.** Casa aportó el **36.24 %** de los ingresos, la mayor participación entre los tipos de propiedad. El resultado respalda priorizar su comercialización.
- **Canal de venta.** Corredor concentró el **72.89 %** de los ingresos. Antes de aumentar la inversión en el canal, conviene revisar sus costos y su conversión.
- **Segmento de cliente.** Primera vez aportó el **62.74 %** de los ingresos.
- **Compras posteriores.** En 2024 se registraron **482 ventas** entre los clientes de la cohorte de marzo de 2023 y **453** entre los de abril. Para evaluar acciones de posventa, conviene comparar cohortes con el mismo tiempo de seguimiento.

**Referencia de cohortes.** Matriz «Ventas mensuales por cohorte», filas `2023-03` y `2023-04`, con `Mes Venta` limitado a `2024-01`–`2024-12`. Los valores cuentan transacciones.
